# Supervised Model Development

This notebook implements the supervised-learning workflow for predicting whether a banking complaint ends with any consumer monetary relief or no relief. The primary source file is `data/processed/final_unsupervised_features_relief_vader.parquet`. Due to size limitations, the data is not available on Github, however, please see the sample_data folder of the repo for a glimpse into 100 examples records.

AI Assistance:
OpenAI ChatGPT was used for code debugging, code generation, code organization,
and code methodological brainstorming. All final modeling, implementation,
validation, commentary, and interpretation were performed and verified by the authors.

## Final Relief Modeling Notebook


In [1]:
# pip install -r ../requirements.txt


## Notebook Overview

The goal is still binary classification:

- `1` = the complaint ended with `Closed with monetary relief`
- `0` = every other company response value in the filtered relief dataset

Relative to the earlier relief draft, this version makes a few deliberate changes:

- updates the positive-class definition so only monetary relief is treated as the positive outcome
- removes geography from the default modeling feature set because prior ablation suggested it was weak or slightly harmful
- adds one lightweight narrative signal through a simple sentiment score
- keeps narrative engineering lightweight so the all-records baseline stays clean and broadly applicable
- keeps all four model families, but uses a more balanced runtime/performance configuration for final development


## Table of Contents

1. [Data Loading and Exploration](#Data-Loading-and-Exploration)
2. [Feature Engineering](#Feature-Engineering)
3. [Train/Test Setup](#Train/Test-Setup)
4. [Model #1 Logistic Regression](#Model-#1-Logistic-Regression)
5. [Model #2 KNN](#Model-#2-KNN)
6. [Model #3 Random Forest](#Model-#3-Random-Forest)
7. [Model #4 Support Vector Machine](#Model-#4-Support-Vector-Machine)
8. [Final Model Selection](#Final-Model-Selection)


### Data Loading and Exploration

We start by validating that the final_unsupervised_features_relief_vader parquet is internally consistent enough for supervised learning. Before training any model, we check for duplicate complaint identifiers, target completeness, date parsing issues, negative date gaps, and the amount of missingness in high-value fields such as the narrative (which will be vital for later models developed using Unsupervised generated features)


In [2]:
import os
from pathlib import Path
import warnings

import joblib
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "final_unsupervised_features_relief_vader.parquet"
ARTIFACT_DIR = PROJECT_ROOT / "data" / "processed"
VISUALS_DIR = PROJECT_ROOT / "visuals"
VISUALS_DIR.mkdir(exist_ok=True)
OUTPUT_TABLES_DIR = PROJECT_ROOT / "output_tables"
OUTPUT_TABLES_DIR.mkdir(exist_ok=True)

MODEL_SCOPE = "all_banking_relief_final"
QUICK_TEST_MODE = False
RANDOM_STATE = 42
CV_FOLDS = 5 if not QUICK_TEST_MODE else 3
MAX_MODEL_ROWS = 120000 if not QUICK_TEST_MODE else 30000
KNN_MAX_TRAIN_ROWS = 15000 if not QUICK_TEST_MODE else 5000
RF_N_ESTIMATORS = 225 if not QUICK_TEST_MODE else 100
N_JOBS = min(4, max(1, (os.cpu_count() or 2) - 1))

assert DATA_PATH.exists(), f"Expected sentiment-enriched relief parquet at {DATA_PATH}"

raw_df = pd.read_parquet(DATA_PATH)

print(f"Loaded {len(raw_df):,} rows and {raw_df.shape[1]} columns from {DATA_PATH.name}")
raw_df.head()


Loaded 397,575 rows and 44 columns from final_unsupervised_features_relief_vader.parquet


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,cleaned_consumer_narrative,processed_narrative,cluster,distance_to_centroid,dominant_topic,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob,topic_8_prob,topic_9_prob,topic_10_prob,topic_11_prob,topic_12_prob,topic_13_prob,topic_14_prob,topic_15_prob,topic_16_prob,topic_17_prob,topic_18_prob,topic_19_prob,narrative_sentiment_score
0,2019-11-18,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,XXXX claimed they delivered a package to my ad...,NaN,DISCOVER BANK,MA,021XX,NaN,Consent provided,Web,2019-11-18,Closed with explanation,True,N/A,3442136,REDACTED claimed they delivered a package to m...,claimed delivered package address never receiv...,11,0.992864,10,0.000602,0.000602,0.000602,0.000602,0.000602,0.000602,0.000602,0.000602,0.000602,0.154693,0.712191,0.000602,0.061553,0.000602,0.000602,0.000602,0.000602,0.000602,0.061924,0.000602,0.4019
1,2020-04-10,Credit card or prepaid card,General-purpose prepaid card,Trouble using the card,Trouble getting information about the card,I got a Brinks Money pre-paid card in the mail...,Company has responded to the consumer and the ...,Netspend Corporation,IL,60657,NaN,Consent provided,Web,2020-04-14,Closed with explanation,True,N/A,3601853,I got a Brinks Money pre-paid card in the mail...,got brink money pre paid card mail assuming un...,3,0.998468,8,0.002174,0.002174,0.276566,0.002174,0.002174,0.002174,0.002174,0.002174,0.503805,0.002174,0.002174,0.002174,0.002174,0.182673,0.002174,0.002174,0.002174,0.002174,0.002174,0.002174,0.0000
2,2019-07-09,Credit card or prepaid card,Store credit card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,On XX/XX/XXXX I was called by a creditor Nelso...,NaN,Nelson Cruz & Associates LLC,TN,37043,NaN,Consent provided,Web,2019-07-09,Closed with explanation,True,N/A,3300820,On REDACTED_DATE I was called by a creditor Ne...,called creditor nelson cruz associate claimed ...,44,0.978543,9,0.000926,0.085328,0.147042,0.000926,0.000926,0.000926,0.000926,0.068550,0.186560,0.386073,0.000926,0.077052,0.000926,0.000926,0.000926,0.000926,0.037359,0.000926,0.000926,0.000926,-0.8002
3,2020-07-10,Credit card or prepaid card,General-purpose credit card or charge card,Trouble using your card,Can't use card to make purchases,Around XX/XX/2020 i XXXX XXXX XXXX opened a cr...,NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,937XX,NaN,Consent provided,Web,2020-07-10,Closed with explanation,True,N/A,3739698,Around REDACTED / REDACTED /2020 i REDACTED RE...,around opened credit card account online capit...,25,0.852135,2,0.000538,0.000538,0.426388,0.000538,0.000538,0.000538,0.000538,0.000538,0.183611,0.320316,0.018011,0.000538,0.000538,0.000538,0.000538,0.000538,0.000538,0.000538,0.000538,0.043609,0.8060
4,2019-06-24,Credit card or prepaid card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,Equifax sent me credit card suggestions to hel...,NaN,"EQUIFAX, INC.",OR,971XX,NaN,Consent provided,Web,2019-06-24,Closed with explanation,True,N/A,3285243,Equifax sent me credit card suggestions to hel...,equifax sent credit card suggestion help impro...,44,0.963603,6,0.002083,0.002083,0.002083,0.002083,0.002083,0.179421,0.783079,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.8591


In [3]:
quality_df = raw_df.copy()
quality_df["Date received"] = pd.to_datetime(quality_df["Date received"], errors="coerce")
quality_df["Date sent to company"] = pd.to_datetime(quality_df["Date sent to company"], errors="coerce")
quality_df["company_lag_days"] = (
    quality_df["Date sent to company"] - quality_df["Date received"]
).dt.days

positive_relief_values = ["Closed with monetary relief"]

quality_summary = pd.DataFrame(
    {
        "dtype": raw_df.dtypes.astype(str),
        "missing_count": raw_df.isna().sum(),
        "missing_pct": (raw_df.isna().mean() * 100).round(2),
        "n_unique": raw_df.nunique(dropna=True),
    }
).sort_values(["missing_pct", "n_unique"], ascending=[False, False])

data_quality_checks = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "duplicate_rows": int(raw_df.duplicated().sum()),
        "duplicate_complaint_ids": int(raw_df["Complaint ID"].duplicated().sum()),
        "null_target_count": int(raw_df["Company response to consumer"].isna().sum()),
        "unparseable_date_received": int(quality_df["Date received"].isna().sum()),
        "unparseable_date_sent": int(quality_df["Date sent to company"].isna().sum()),
        "negative_lag_rows": int((quality_df["company_lag_days"] < 0).sum()),
        "narrative_missing_pct": round(raw_df["Consumer complaint narrative"].isna().mean() * 100, 2),
        "target_relief_pct": round(raw_df["Company response to consumer"].isin(positive_relief_values).mean() * 100, 3),
    },
    name="value",
)

display(data_quality_checks.to_frame())
display(quality_summary.head(15))
display(quality_df["company_lag_days"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame().T)


,value
rows,397575.000
columns,44.000
duplicate_rows,0.000
duplicate_complaint_ids,0.000
null_target_count,0.000
unparseable_date_received,0.000
unparseable_date_sent,0.000
negative_lag_rows,0.000
narrative_missing_pct,0.000
target_relief_pct,15.719


,dtype,missing_count,missing_pct,n_unique
Tags,str,314638,79.14,3
Company public response,str,208623,52.47,10
Consumer complaint narrative,str,0,0.00,397575
Complaint ID,int64,0,0.00,397575
cleaned_consumer_narrative,str,0,0.00,397575
processed_narrative,str,0,0.00,397575
distance_to_centroid,float64,0,0.00,397214
topic_2_prob,float64,0,0.00,397115
topic_12_prob,float64,0,0.00,397115
topic_1_prob,float64,0,0.00,397114


,count,mean,std,min,1%,5%,50%,95%,99%,max
company_lag_days,397575.0,1.187532,6.251914,0.0,0.0,0.0,0.0,6.0,33.0,446.0


The quality check still does its usual job here, but the design implications are more focused in the final notebook:

- the target rate now reflects monetary relief only, which is the right binary framing for this final notebook
- the narrative is now a complete subset, so we only keep lightweight derived features that are robust to missing text
- very sparse fields remain poor candidates for prediction use, so the feature set stays selective rather than large and broad
- the final notebook is intentionally conservative about feature growth: we would rather add a few strong signals than flood the models with weak ones, which may be the case with some sub-issue fields and clearly was the case with the ZIP geography looked at in the old relief notebook.


In [4]:
product_summary = (
    raw_df.groupby("Product", dropna=False)
    .agg(
        complaints=("Complaint ID", "count"),
        relief_rate=("Company response to consumer", lambda s: s.isin(["Closed with monetary relief"]).mean()),
        narrative_available=("Consumer complaint narrative", lambda s: s.notna().mean()),
    )
    .sort_values("complaints", ascending=False)
)
product_summary[["relief_rate", "narrative_available"]] = (
    product_summary[["relief_rate", "narrative_available"]] * 100
).round(2)

display(product_summary)
display(raw_df["Company response to consumer"].value_counts(dropna=False).rename("count").to_frame())

high_null_columns = quality_summary.loc[quality_summary["missing_pct"] >= 20, ["missing_pct", "n_unique"]]
display(high_null_columns)


,complaints,relief_rate,narrative_available
Product,,,
Checking or savings account,170983,15.43,100.0
Credit card or prepaid card,100503,19.06,100.0
Credit card,89765,18.11,100.0
Mortgage,36324,1.93,100.0


,count
Company response to consumer,
Closed with explanation,302126
Closed with monetary relief,62496
Closed with non-monetary relief,32953


,missing_pct,n_unique
Tags,79.14,3
Company public response,52.47,10


Overall, the initial quality check shows a few important findings that shape the rest of the notebook:

- The processed banking dataset is large enough for supervised learning at almost 400k records even with our filtered down subset.

- `Complaint ID` appears unique and the date fields parse cleanly, which reduces the need for dedup logic or date parsing.

- The filtered relief dataset is much more balanced than the untimely response dataset we worked with originally, so we use explicit imbalance handling and tractability controls in this notebook because the relief dataset is larger and may still benefit from rebalancing depending on the final class mix.

- Narrative coverage is still meaningful and while complete, still varies greatly, which motivates combining simple narrative indicators with structured complaint context.

- Missingness is concentrated in fields like `Tags` and `Company public response`, which make them unlikely to be useful. Furthermore, `Consumer complaint narrative` and `Sub-issue` also contain missing values, so they might end up being common points of failure in our results.

The product summary also confirms that this is a banking problem rather than a single product classification task. That variety is one reason we preserve `Product`, `Issue`, channel, and company context as predictive inputs.


### Feature Engineering

We intentionally exclude fields that are likely to leak post-submission or post-resolution information into the prediction task. In particular, `Date sent to company`, `Timely response?`, `Company public response`, and `Consumer disputed?` are better suited for diagnostics than for prediction of the company monetary relief vs no monetary relief response outcome.

Our feature design tries to balance predictive power with practicality. If a feature is only known after the complaint has already been processed by the company, we exclude it from model training even if it would make prediction easier.

Narrative information in this notebook is intentionally kept simple:

- whether a narrative is present
- basic narrative length measures such as character and word count

More advanced narrative modeling is intentionally left out of this notebook so it does not overlap with the separate unsupervised clustering and feature generation notebook results. We will have a separate supervised run on the more complete narrative subset to see how much better performance we get when data is complete.


In [5]:
def make_one_hot_encoder(dense=False):
    kwargs = {"handle_unknown": "ignore"}
    try:
        return OneHotEncoder(sparse_output=not dense, **kwargs)
    except TypeError:
        return OneHotEncoder(sparse=not dense, **kwargs)


POSITIVE_RELIEF_VALUES = ["Closed with monetary relief"]


def score_estimator(estimator, X_test, y_test):
    if hasattr(estimator, "predict_proba"):
        y_score = estimator.predict_proba(X_test)[:, 1]
    else:
        y_score = estimator.decision_function(X_test)
    y_pred = estimator.predict(X_test)
    return {
        "holdout_accuracy": accuracy_score(y_test, y_pred),
        "holdout_average_precision": average_precision_score(y_test, y_score),
        "holdout_roc_auc": roc_auc_score(y_test, y_score),
        "holdout_recall": recall_score(y_test, y_pred, zero_division=0),
        "holdout_precision": precision_score(y_test, y_pred, zero_division=0),
        "holdout_f1": f1_score(y_test, y_pred, zero_division=0),
        "holdout_balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "y_score": y_score,
        "y_pred": y_pred,
    }


def build_results_row(model_name, search, X_test, y_test):
    best_idx = search.best_index_
    metric_names = [
        "accuracy",
        "average_precision",
        "roc_auc",
        "recall",
        "precision",
        "f1",
        "balanced_accuracy",
    ]
    row = {
        "model_family": model_name,
        "best_params": search.best_params_,
    }
    for metric in metric_names:
        row[f"cv_mean_{metric}"] = search.cv_results_[f"mean_test_{metric}"][best_idx]
        row[f"cv_std_{metric}"] = search.cv_results_[f"std_test_{metric}"][best_idx]

    holdout_scores = score_estimator(search.best_estimator_, X_test, y_test)
    row.update({k: v for k, v in holdout_scores.items() if not k.startswith("y_")})
    return row, holdout_scores


leakage_columns = [
    "Timely response?",
    "Date sent to company",
    "Company response to consumer",
    "Company public response",
    "Consumer disputed?",
    "Consumer consent provided?",
]

model_df = raw_df.copy()
model_df["Date received"] = pd.to_datetime(model_df["Date received"], errors="coerce")
model_df["target_relief"] = model_df["Company response to consumer"].isin(POSITIVE_RELIEF_VALUES).astype(int)

text_series = model_df["Consumer complaint narrative"].fillna("")
model_df["narrative_present"] = text_series.str.len().gt(0).astype(int)
model_df["narrative_char_count"] = text_series.str.len()
model_df["narrative_word_count"] = text_series.str.split().str.len().fillna(0)

if "narrative_sentiment_score" not in model_df.columns:
    raise KeyError("Expected precomputed 'narrative_sentiment_score' column. Run notebooks/02b_narrative_sentiment.ipynb first.")

model_df["received_year"] = model_df["Date received"].dt.year
model_df["received_month"] = model_df["Date received"].dt.month
model_df["received_quarter"] = model_df["Date received"].dt.quarter
model_df["received_dayofweek"] = model_df["Date received"].dt.dayofweek
model_df["received_day"] = model_df["Date received"].dt.day
model_df["zip3"] = (
    model_df["ZIP code"].fillna("").astype(str).str.extract(r"(\d{3})", expand=False).fillna("missing")
)

# Limiting our categories to prevent unnecessary low-importance categories from making model performance worse with little to no benefit
top_category_limits = {
    "Company": 60 if QUICK_TEST_MODE else 100,
    "Sub-product": 20 if QUICK_TEST_MODE else 35,
    "Issue": 25 if QUICK_TEST_MODE else 45,
    "Sub-issue": 35 if QUICK_TEST_MODE else 70,
}
for col, top_n in top_category_limits.items():
    values = model_df[col].fillna("missing").astype(str)
    keep = set(values.value_counts().head(top_n).index)
    model_df[col] = values.where(values.isin(keep), other="__OTHER__")

model_df["Submitted via"] = model_df["Submitted via"].fillna("missing").astype(str)
model_df["Product"] = model_df["Product"].fillna("missing").astype(str)

usable_columns = [col for col in model_df.columns if col not in leakage_columns]
model_df = model_df[usable_columns].dropna(subset=["Date received", "target_relief"]).copy()

if len(model_df) > MAX_MODEL_ROWS:
    _, model_df = train_test_split(
        model_df,
        test_size=MAX_MODEL_ROWS,
        stratify=model_df["target_relief"],
        random_state=RANDOM_STATE,
    )

display(model_df.head())
print(f"Modeling rows after filtering: {len(model_df):,}")
print(f"Relief class rate: {model_df['target_relief'].mean() * 100:.3f}%")


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company,State,ZIP code,Tags,Submitted via,Complaint ID,cleaned_consumer_narrative,processed_narrative,cluster,distance_to_centroid,dominant_topic,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob,topic_8_prob,topic_9_prob,topic_10_prob,topic_11_prob,topic_12_prob,topic_13_prob,topic_14_prob,topic_15_prob,topic_16_prob,topic_17_prob,topic_18_prob,topic_19_prob,narrative_sentiment_score,target_relief,narrative_present,narrative_char_count,narrative_word_count,received_year,received_month,received_quarter,received_dayofweek,received_day,zip3
3098,2020-01-09,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,"On XX/XX/2020, I went into my bank account to ...",WELLS FARGO & COMPANY,CA,92672,NaN,Web,3490683,"On REDACTED / REDACTED /2020, I went into my b...",went bank account check statement noticed cred...,6,0.965072,2,0.000459,0.118154,0.455229,0.000459,0.000459,0.205911,0.112035,0.000459,0.000459,0.000459,0.000459,0.011033,0.000459,0.000459,0.048028,0.000459,0.000459,0.000459,0.000459,0.043647,-0.3395,1,1,1356,252,2020,1,1,3,9,926
103540,2018-02-08,Checking or savings account,Checking account,Opening an account,Didn't receive terms that were advertised,"On XX/XX/XXXX, I opened a Citibank Account Pac...","CITIBANK, N.A.",CA,90024,NaN,Web,2808333,"On REDACTED_DATE , I opened a Citibank Account...",opened citibank account package checking accou...,42,0.886301,15,0.000532,0.000532,0.000532,0.000532,0.295650,0.000532,0.000532,0.000532,0.000532,0.000532,0.000532,0.000532,0.000532,0.000532,0.000532,0.681927,0.000532,0.013381,0.000532,0.000532,0.9531,1,1,1195,186,2018,2,1,3,8,900
270594,2024-01-29,Mortgage,Conventional home mortgage,Trouble during payment process,Trying to communicate with the company to fix ...,In XX/XX/2023 we applied for a principal curta...,"United Shore Financial Services, LLC",TN,374XX,NaN,Web,8239106,In REDACTED / REDACTED /2023 we applied for a ...,applied principal curtailment loan recast appr...,26,0.982504,13,0.076498,0.146219,0.136955,0.098328,0.095437,0.000240,0.000240,0.000240,0.000240,0.000240,0.000240,0.000240,0.000240,0.443198,0.000240,0.000240,0.000240,0.000240,0.000240,0.000240,0.3182,0,1,2649,463,2024,1,1,0,29,374
74997,2023-11-27,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,Been a customer for almost 2 years. Had a repl...,__OTHER__,CO,80205,NaN,Web,7909093,Been a customer for almost 2 years. Had a repl...,customer almost year replacement card stolen u...,18,0.959723,8,0.000289,0.000289,0.237072,0.000289,0.000289,0.120216,0.000289,0.000289,0.282905,0.000289,0.000289,0.126285,0.169310,0.000289,0.000289,0.060166,0.000289,0.000289,0.000289,0.000289,-0.9224,0,1,2217,399,2023,11,4,0,27,802
297136,2023-10-08,Mortgage,Conventional home mortgage,Applying for a mortgage or refinancing an exis...,__OTHER__,"I had received a letter from state, three year...",NEW YORK COMMUNITY BANCORP INC,CA,90604,NaN,Web,7661760,"I had received a letter from state, three year...",received letter state three year brother owned...,32,0.981038,3,0.000362,0.000362,0.000362,0.597497,0.000362,0.000362,0.000362,0.000362,0.044476,0.000362,0.000362,0.000362,0.000362,0.246530,0.105700,0.000362,0.000362,0.000362,0.000362,0.000362,0.9912,0,1,2279,417,2023,10,4,6,8,906


Modeling rows after filtering: 120,000
Relief class rate: 15.719%


A few design choices here:

- The final notebook keeps simple narrative features only: presence, size, and a preprocessed sentiment score.
- Topic clusters are intentionally left out of this all-records baseline and reserved for the separate narrative-focused notebook.
- Geography is deliberately removed from the main final feature set because prior ablation suggested it added cost without enough signal.
- We still use stratified train/test splitting because preserving the class ratio is good practice even when the binary target is not extremely skewed.
- The all-records final notebook is meant to be the structured baseline that can score every complaint, whether or not a narrative exists.
- We stay careful with missingness: categorical features are explicitly labeled as `missing`, while numeric features use median imputation inside each training fold.


### Train/Test Setup

We use a stratified split and 5-fold stratified cross-validation and report accuracy, precision, recall, F1, ROC AUC, balanced accuracy, and average precision for each model. Cross-fold model selection still uses F1, but we use stratified splits plus targeted resampling for the distance based model because this relief dataset is larger and the monetary relief positive class may still benefit from balancing support.

For the final notebook, the non-tree models also get a slightly broader search space than the quick relief draft did, but the grids are still intentionally selective so runtime grows in a controlled way rather than exploding from low-value combinations.


In [6]:
target_col = "target_relief"

structured_numeric = [
    "received_year",
    "received_month",
    "received_quarter",
    "received_dayofweek",
    "received_day",
    "narrative_present",
    "narrative_char_count",
    "narrative_word_count",
    "narrative_sentiment_score",
]

shared_topic_features = []

logistic_categorical = [
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Company",
    "Submitted via",
] + shared_topic_features

knn_categorical = ["Product", "Issue", "Submitted via"] + shared_topic_features
rf_categorical = [
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Company",
    "Submitted via",
] + shared_topic_features

logistic_features = logistic_categorical + structured_numeric
knn_features = knn_categorical + structured_numeric
rf_features = rf_categorical + structured_numeric

train_df, test_df = train_test_split(
    model_df,
    test_size=0.2,
    stratify=model_df[target_col],
    random_state=RANDOM_STATE,
)

scoring = {
    "accuracy": "accuracy",
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
    "recall": "recall",
    "precision": "precision",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy",
}
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Train relief rate: {train_df[target_col].mean() * 100:.3f}%")
print(f"Test relief rate: {test_df[target_col].mean() * 100:.3f}%")
print(f"Grid-search workers: {N_JOBS}")
print(f"Quick test mode: {QUICK_TEST_MODE}")
print(f"Max modeling rows: {MAX_MODEL_ROWS}")
print(f"KNN train-row cap: {KNN_MAX_TRAIN_ROWS}")
print(f"Logistic grid points: {len(logistic_grid) if 'logistic_grid' in globals() else 'defined later'}")



Train rows: 96,000
Test rows: 24,000
Train relief rate: 15.719%
Test relief rate: 15.721%
Grid-search workers: 4
Quick test mode: False
Max modeling rows: 120000
KNN train-row cap: 15000
Logistic grid points: defined later


### Model #1 Logistic Regression

Logistic regression remains our strongest linear baseline. In the final notebook, it gets the cleaner feature set, a sentiment score, and company context. That gives it a fair chance to compete without turning the run into a huge sparse-matrix exercise.


In [7]:
logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("onehot", make_one_hot_encoder(dense=False)),
                ]
            ),
            logistic_categorical,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            structured_numeric,
        ),
    ]
)

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", logistic_preprocessor),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=900,
                solver="saga",
                tol=5e-4,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_grid = {
    "model__C": [0.15, 0.35, 0.75, 1.5, 3.0],
}

logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=N_JOBS,
    verbose=1,
)

X_train_log = train_df[logistic_features]
y_train_log = train_df[target_col]
X_test_log = test_df[logistic_features]
y_test_log = test_df[target_col]

logistic_search.fit(X_train_log, y_train_log)
logistic_row, logistic_scores = build_results_row("Logistic Regression", logistic_search, X_test_log, y_test_log)

logistic_holdout = test_df[["Complaint ID", "Product", "Issue", "Company", "narrative_present", target_col]].copy()
logistic_holdout["model_family"] = "Logistic Regression"
logistic_holdout["score"] = logistic_scores["y_score"]
logistic_holdout["prediction"] = logistic_scores["y_pred"]

display(pd.DataFrame([logistic_row]).T)


Fitting 5 folds for each of 5 candidates, totalling 25 fits


,0
model_family,Logistic Regression
best_params,{'model__C': 0.15}
cv_mean_accuracy,0.684229
cv_std_accuracy,0.002969
cv_mean_average_precision,0.381621
cv_std_average_precision,0.004909
cv_mean_roc_auc,0.78903
cv_std_roc_auc,0.003402
cv_mean_recall,0.767462
cv_std_recall,0.010218


### Model #2 KNN

KNN is still included for family coverage, but it is intentionally kept compact because distance-based models become expensive quickly once categorical one-hot features expand. We therefore use a reduced feature set, a capped training sample, and SMOTE inside the pipeline so the comparison stays fair without dominating runtime.


In [8]:
if KNN_MAX_TRAIN_ROWS is not None and len(train_df) > KNN_MAX_TRAIN_ROWS:
    _, knn_train_df = train_test_split(
        train_df,
        test_size=KNN_MAX_TRAIN_ROWS,
        stratify=train_df[target_col],
        random_state=RANDOM_STATE,
    )
else:
    knn_train_df = train_df.copy()

knn_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("onehot", make_one_hot_encoder(dense=True)),
                ]
            ),
            knn_categorical,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            structured_numeric,
        ),
    ]
)

knn_pipeline = ImbPipeline(
    steps=[
        ("preprocessor", knn_preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.55, k_neighbors=3)),
        ("model", KNeighborsClassifier()),
    ]
)

knn_grid = {
    "smote__sampling_strategy": [0.45, 0.55, 0.65],
    "model__n_neighbors": [21, 31, 51, 71],
    "model__weights": ["distance"],
    "model__p": [1, 2],
}

knn_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=N_JOBS,
    verbose=1,
)

X_train_knn = knn_train_df[knn_features]
y_train_knn = knn_train_df[target_col]
X_test_knn = test_df[knn_features]
y_test_knn = test_df[target_col]

knn_search.fit(X_train_knn, y_train_knn)
knn_row, knn_scores = build_results_row("KNN", knn_search, X_test_knn, y_test_knn)

knn_holdout = test_df[["Complaint ID", "Product", "Issue", "Company", "narrative_present", target_col]].copy()
knn_holdout["model_family"] = "KNN"
knn_holdout["score"] = knn_scores["y_score"]
knn_holdout["prediction"] = knn_scores["y_pred"]

display(pd.DataFrame([knn_row]).T)


Fitting 5 folds for each of 24 candidates, totalling 120 fits


,0
model_family,KNN
best_params,"{'model__n_neighbors': 71, 'model__p': 1, 'mod..."
cv_mean_accuracy,0.736133
cv_std_accuracy,0.011617
cv_mean_average_precision,0.260904
cv_std_average_precision,0.014911
cv_mean_roc_auc,0.673684
cv_std_roc_auc,0.013165
cv_mean_recall,0.408403
cv_std_recall,0.010984


### Model #3 Random Forest

Random Forest remains the main candidate for the final supervised workflow because it can mix nonlinear interactions across complaint context, company, and lightweight narrative indicators without forcing aggressive preprocessing assumptions. The final version gives it a broader hyperparameter search.


In [9]:
rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("onehot", make_one_hot_encoder(dense=False)),
                ]
            ),
            rf_categorical,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            ),
            structured_numeric,
        ),
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=RF_N_ESTIMATORS,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
            ),
        ),
    ]
)

rf_grid = {
    "model__max_depth": [14, 22, None],
    "model__min_samples_leaf": [2, 5, 10],
    "model__min_samples_split": [5, 10],
    "model__max_features": ["sqrt", 0.5],
}

rf_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=N_JOBS,
    verbose=1,
)

X_train_rf = train_df[rf_features]
y_train_rf = train_df[target_col]
X_test_rf = test_df[rf_features]
y_test_rf = test_df[target_col]

rf_search.fit(X_train_rf, y_train_rf)
rf_row, rf_scores = build_results_row("Random Forest", rf_search, X_test_rf, y_test_rf)

rf_holdout = test_df[["Complaint ID", "Product", "Issue", "Company", "narrative_present", target_col]].copy()
rf_holdout["model_family"] = "Random Forest"
rf_holdout["score"] = rf_scores["y_score"]
rf_holdout["prediction"] = rf_scores["y_pred"]

display(pd.DataFrame([rf_row]).T)


Fitting 5 folds for each of 36 candidates, totalling 180 fits


,0
model_family,Random Forest
best_params,"{'model__max_depth': None, 'model__max_feature..."
cv_mean_accuracy,0.781844
cv_std_accuracy,0.001554
cv_mean_average_precision,0.407231
cv_std_average_precision,0.00918
cv_mean_roc_auc,0.797355
cv_std_roc_auc,0.002591
cv_mean_recall,0.575149
cv_std_recall,0.007142


### Model #4 Support Vector Machine

The linear SVM gives us another high-dimensional margin-based baseline. Like logistic regression, it benefits from the reduced final feature set and sentiment score, but it avoids the probability calibration overhead by using the raw decision margin for ranking metrics.


In [10]:
svm_categorical = [
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Company",
    "Submitted via",
] + shared_topic_features
svm_features = svm_categorical + structured_numeric

svm_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("onehot", make_one_hot_encoder(dense=False)),
                ]
            ),
            svm_categorical,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            structured_numeric,
        ),
    ]
)

svm_pipeline = Pipeline(
    steps=[
        ("preprocessor", svm_preprocessor),
        ("model", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, dual="auto", max_iter=10000, tol=5e-4)),
    ]
)

svm_grid = {
    "model__C": [0.1, 0.25, 0.5, 1.0, 2.0],
}

svm_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=N_JOBS,
    verbose=1,
)

X_train_svm = train_df[svm_features]
y_train_svm = train_df[target_col]
X_test_svm = test_df[svm_features]
y_test_svm = test_df[target_col]

svm_search.fit(X_train_svm, y_train_svm)
svm_row, svm_scores = build_results_row("Support Vector Machine", svm_search, X_test_svm, y_test_svm)

svm_holdout = test_df[["Complaint ID", "Product", "Issue", "Company", "narrative_present", target_col]].copy()
svm_holdout["model_family"] = "Support Vector Machine"
svm_holdout["score"] = svm_scores["y_score"]
svm_holdout["prediction"] = svm_scores["y_pred"]

display(pd.DataFrame([svm_row]).T)


Fitting 5 folds for each of 5 candidates, totalling 25 fits


,0
model_family,Support Vector Machine
best_params,{'model__C': 0.1}
cv_mean_accuracy,0.676073
cv_std_accuracy,0.001689
cv_mean_average_precision,0.38152
cv_std_average_precision,0.004896
cv_mean_roc_auc,0.789126
cv_std_roc_auc,0.003563
cv_mean_recall,0.777203
cv_std_recall,0.010104


### Final Model Selection

The final comparison is still ranked by cross-validated F1 so that model selection stays consistent with the project requirement. We still report the other metrics, but the main decision rule remains F1.


In [11]:
results_df = pd.DataFrame([logistic_row, knn_row, rf_row, svm_row]).sort_values(
    by="cv_mean_f1", ascending=False
).reset_index(drop=True)

comparison_columns = [
    "model_family",
    "cv_mean_accuracy",
    "cv_std_accuracy",
    "cv_mean_average_precision",
    "cv_std_average_precision",
    "cv_mean_roc_auc",
    "cv_std_roc_auc",
    "cv_mean_recall",
    "cv_std_recall",
    "cv_mean_precision",
    "cv_std_precision",
    "cv_mean_f1",
    "cv_std_f1",
    "holdout_accuracy",
    "holdout_average_precision",
    "holdout_roc_auc",
    "holdout_recall",
    "holdout_precision",
    "holdout_f1",
]
display(results_df[comparison_columns])

searches = {
    "Logistic Regression": logistic_search,
    "KNN": knn_search,
    "Random Forest": rf_search,
    "Support Vector Machine": svm_search,
}
holdout_frames = {
    "Logistic Regression": logistic_holdout,
    "KNN": knn_holdout,
    "Random Forest": rf_holdout,
    "Support Vector Machine": svm_holdout,
}

best_model_name = results_df.loc[0, "model_family"]
best_search = searches[best_model_name]
best_holdout = holdout_frames[best_model_name].sort_values("score", ascending=False).reset_index(drop=True)

print(f"Selected development model: {best_model_name}")
print(f"Best params: {best_search.best_params_}")

comparison_path = OUTPUT_TABLES_DIR / "final_relief_supervised_model_comparison_clean.csv"
visuals_comparison_path = OUTPUT_TABLES_DIR / "03_final_model_comparison_clean.csv"
holdout_path = ARTIFACT_DIR / "final_relief_best_model_holdout_predictions_clean.parquet"
model_path = ARTIFACT_DIR / "best_final_relief_response_model_clean.joblib"

results_df.to_csv(comparison_path, index=False)
results_df[comparison_columns].round(4).to_csv(visuals_comparison_path, index=False)
best_holdout.to_parquet(holdout_path, index=False)
joblib.dump(best_search.best_estimator_, model_path)

print(f"Saved comparison table to {comparison_path}")
print(f"Saved GitHub-friendly comparison table to {visuals_comparison_path}")
print(f"Saved holdout predictions to {holdout_path}")
print(f"Saved trained model to {model_path}")

best_holdout.head(10)


,model_family,cv_mean_accuracy,cv_std_accuracy,cv_mean_average_precision,cv_std_average_precision,cv_mean_roc_auc,cv_std_roc_auc,cv_mean_recall,cv_std_recall,cv_mean_precision,cv_std_precision,cv_mean_f1,cv_std_f1,holdout_accuracy,holdout_average_precision,holdout_roc_auc,holdout_recall,holdout_precision,holdout_f1
0,Random Forest,0.781844,0.001554,0.407231,0.009180,0.797355,0.002591,0.575149,0.007142,0.373920,0.002768,0.453189,0.003696,0.781625,0.408751,0.798099,0.583886,0.375043,0.456722
1,Logistic Regression,0.684229,0.002969,0.381621,0.004909,0.789030,0.003402,0.767462,0.010218,0.301701,0.003045,0.433123,0.004438,0.681667,0.391520,0.790104,0.763583,0.299200,0.429936
2,Support Vector Machine,0.676073,0.001689,0.381520,0.004896,0.789126,0.003563,0.777203,0.010104,0.297182,0.002313,0.429954,0.003842,0.674542,0.392431,0.790101,0.772595,0.295399,0.427388
3,KNN,0.736133,0.011617,0.260904,0.014911,0.673684,0.013165,0.408403,0.010984,0.273715,0.016175,0.327630,0.014573,0.732875,0.256963,0.675803,0.412404,0.270609,0.326788


Selected development model: Random Forest
Best params: {'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 10}
Saved comparison table to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\output_tables\final_relief_supervised_model_comparison_clean.csv
Saved GitHub-friendly comparison table to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\output_tables\03_final_model_comparison_clean.csv
Saved holdout predictions to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\data\processed\final_relief_best_model_holdout_predictions_clean.parquet
Saved trained model to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\data\processed\best_final_relief_response_model_clean.joblib


,Complaint ID,Product,Issue,Company,narrative_present,target_relief,model_family,score,prediction
0,11203612,Credit card,Problem with a purchase shown on your statement,"BANK OF AMERICA, NATIONAL ASSOCIATION",1,1,Random Forest,0.890367,1
1,17890213,Credit card,Problem with a purchase shown on your statement,"BANK OF AMERICA, NATIONAL ASSOCIATION",1,1,Random Forest,0.884435,1
2,2830224,Checking or savings account,Managing an account,"BANK OF AMERICA, NATIONAL ASSOCIATION",1,1,Random Forest,0.884208,1
3,9528325,Credit card,Problem with a purchase shown on your statement,"BANK OF AMERICA, NATIONAL ASSOCIATION",1,1,Random Forest,0.877649,1
4,7168942,Credit card or prepaid card,Fees or interest,"BANK OF AMERICA, NATIONAL ASSOCIATION",1,0,Random Forest,0.871965,1
5,7007621,Credit card or prepaid card,Fees or interest,"Bread Financial Holdings, Inc.",1,1,Random Forest,0.866818,1
6,5458617,Credit card or prepaid card,Fees or interest,"BANK OF AMERICA, NATIONAL ASSOCIATION",1,0,Random Forest,0.866505,1
7,11227989,Credit card,Fees or interest,"Bread Financial Holdings, Inc.",1,1,Random Forest,0.864314,1
8,10101888,Credit card,Fees or interest,"Bread Financial Holdings, Inc.",1,1,Random Forest,0.864111,1
9,8762714,Credit card,Problem with a purchase shown on your statement,"BANK OF AMERICA, NATIONAL ASSOCIATION",1,1,Random Forest,0.860561,1
